In [1]:
import numpy as np

from scipy.linalg import cho_factor, cho_solve, cholesky, solve_triangular
from skglm import WeightedLasso

# Linearly Transformed Sparse Probabilistic PCA (Demo)

## Model

### Helpers

In [2]:
def is_pd(A):
    """
    Check if matrix is positive definite.
    """
    S = (A + A.T) / 2
    try:
        np.linalg.cholesky(S)
        return True
    except np.linalg.LinAlgError:
        return False
    
def ensure_pd(A, eps=1e-6, verbose=False):
    """
    Check if matrix is positive definite, and if not clip eigenvalues.
    """
    if is_pd(A):
        return A
    
    if verbose:
        print("Not already PD. Clipping eigenvalues.")
        
    eigvals, eigvecs = np.linalg.eigh(A)
    eigvals = np.maximum(eigvals, eps)
    return eigvecs @ np.diag(eigvals) @ eigvecs.T

In [3]:
def estimate_sigma_hat(ys, Ps, V):
    """
    Fast unconstrained least-squares estimator of effective latent covariance Σ̂(V). 
    """
    q = V.shape[1]
    q2 = q * q

    G = np.zeros((q2, q2))
    h = np.zeros(q2)

    for y, P in zip(ys, Ps):
        A = P @ V
        C = A.T @ A
        Ay = A.T @ y
        D = np.outer(Ay, Ay) - C

        G += np.kron(C, C)
        h += D.reshape(-1)

    s, *_ = np.linalg.lstsq(G, h, rcond=None)
    Sigma_hat = s.reshape(q, q)
    Sigma_hat = 0.5 * (Sigma_hat + Sigma_hat.T)
    
    return Sigma_hat

### Sparse Model: Linearly Transformed Sparse PPCA

In [4]:
def fit_lin_sparse_ppca(P_obs, y_obs, mu, Sigma_diag, W_init, lam, tol=1e-4, max_iters=1000):
    """
    Estimate W via EM-algorithm for Linearly Transformed Sparse PPCA
    
        P_obs: list of projection matrices
        y_obs: list of observed vectors
        Sigma_diag: list of 1D arrays, each equal to diag(Σ_n)
    """
    
    N = len(y_obs)
    d, q = W_init.shape
    
    W = W_init.copy()

    # construct whitened y and P 
    y = [None] * N
    P = [None] * N
    for n in range(N):
        sigma_inv_sqrt = np.reciprocal(np.sqrt(Sigma_diag[n]))
        y[n] = sigma_inv_sqrt * y_obs[n]
        P[n] = sigma_inv_sqrt[:, np.newaxis] * P_obs[n] 
    
    # build reusable items
    I_q = np.eye(q)
    centered = [y[n] - P[n] @ mu for n in range(N)]  # (y_n - P_n μ)
    P_nTP_n_ii = [np.sum(P[n] ** 2, axis=0) for n in range(N)]  # (P_nᵀ P_n)_ii
    P_nTcentered = [P[n].T @ centered[n] for n in range(N)]  # P_nᵀ (y_n - P_n μ)
    
    observed = np.count_nonzero(P_nTP_n_ii, axis=0) > 0
    unobserved_features = np.where(~observed)[0]
    if len(unobserved_features):
        print(f"Note: features {unobserved_features} are never observed.")

    M_n_inv = np.zeros((max_iters, N, q, q))  # M_n^{-1} = Var[z_n | y_n, W]
    EzzT_post = np.zeros((max_iters, N, q, q))  # E[z_n z_n^T | y_n, W]
    W_history = np.zeros((max_iters, d, q))  # W estimate at each EM iteration
    
    for it in range(max_iters):
            
        Q = np.zeros((d, q, q))
        b = np.zeros((d, q))
        
        for n in range(N):
            # M_n = (Wᵀ P_nᵀ P_n W) + I_q
            PW = P[n] @ W
            M_n = (PW.T @ PW) + I_q

            M_n_inv[it, n] = cho_solve(cho_factor(M_n), I_q)

            # <z_n> = M_n^{-1} Wᵀ P_nᵀ (y_n - P_n μ)
            # note: the below BLAS-3 is faster than two BLAS-2
            Ez_n_post = (M_n_inv[it, n] @ PW.T) @ centered[n]

            # <z_n z_nᵀ> = Ez Ezᵀ + M_inv
            EzzT_post[it, n] = np.outer(Ez_n_post,Ez_n_post) + M_n_inv[it, n]

            # update each Q[i], b[i]
            Q += P_nTP_n_ii[n][:, np.newaxis, np.newaxis] * EzzT_post[it, n]
            b += Ez_n_post * P_nTcentered[n][:, np.newaxis]
            
        
        W_new = np.zeros((d, q))
        for i in range(d):
            
            if not observed[i]:  # feature i not observed (degenerate case)
                continue  # leave W_new[i, :] = 0 
        
            X_star_i = cholesky(Q[i])  # upper triangular cholesky factor U s.t. Q_i = UᵀU
            Y_star_i = solve_triangular(X_star_i.T, b[i], lower=True)

            # normalize columns of X_star_i for preconditioning
            G_inv = np.reciprocal(np.linalg.norm(X_star_i, axis=0))
            X_star_i *= G_inv  

            regr = WeightedLasso(alpha=(lam/q), weights=G_inv, fit_intercept=False)
            regr.fit(X_star_i, Y_star_i)
            W_new[i, :] = regr.coef_ * G_inv  # map back from preconditioned 
            
        # whitening-based normalization to address scale invariance
        HatSigmaW = estimate_sigma_hat(y, P, W_new)
        HatSigmaW = ensure_pd(HatSigmaW)
        R = cholesky(HatSigmaW)
        W_new = W_new @ R
            
        # check convergence    
        W_history[it] = W_new
        if (np.linalg.norm(W_new - W)) < tol:
            print(f'Converged. Iterations = {it+1}.')
            return W_new, W_history[:it+1], M_n_inv[:it+1], EzzT_post[:it+1] 
        
        W = W_new
        
    print(f'Failed to converge. Iterations = {it+1}.')
    return W, W_history, M_n_inv, EzzT_post

### Baseline: Linearly Transformed PPCA

In [5]:
def fit_lin_ppca(P_obs, y_obs, mu, Sigma_diag, W_init, tol=1e-4, max_iters=1000):
    """
    Linearly Transformed PPCA, adapted from Tipping & Bishop 1999 to include linear transformations
    
    Estimates W via EM-algorithm
    
        P_obs: list of projection matrices P_tilde_n
        y_obs: list of observed vectors y_tilde_n
        Sigma_diag: list of 1D arrays, each equal to diag(Σ_n)
    """

    N = len(y_obs)
    d, q = W_init.shape
    
    W = W_init.copy()
    
    # construct whitened y and P 
    y = [None] * N
    P = [None] * N
    for n in range(N):
        sigma_inv_sqrt = np.reciprocal(np.sqrt(Sigma_diag[n]))
        y[n] = sigma_inv_sqrt * y_obs[n]
        P[n] = sigma_inv_sqrt[:, np.newaxis] * P_obs[n] 

    # build reusable items
    I_q = np.eye(q)
    centered = [y[n] - P[n] @ mu for n in range(N)]  # (y_n - P_n μ)
    P_nTcentered = [P[n].T @ centered[n] for n in range(N)]  # P_nᵀ (y_n - P_n μ)
    P_nTP_n_ii = np.array([np.sum(P[n] ** 2, axis=0) for n in range(N)])  # (P_nᵀ P_n)_ii 
    
    unobserved_features = np.where(np.count_nonzero(P_nTP_n_ii, axis=0) == 0)[0]
    if len(unobserved_features):
        print("Note: Features ", unobserved_features, " are never observed.")
    
    for it in range(max_iters):

        # build W row-wise
        Numerator = np.zeros((d, q))
        EzzT = np.zeros((N, q, q))  # for denominator calculation

        for n in range(N):
            # M_n = (Wᵀ P_nᵀ P_n W) + I_q
            PW = P[n] @ W
            M_n = (PW.T @ PW) + I_q

            M_n_inv = cho_solve(cho_factor(M_n), I_q)

            # <z_n> = M_inv Wᵀ P_nᵀ (y_n - P_n μ)
            Ez_n = (M_n_inv @ PW.T) @ centered[n]

            # <z_n z_nᵀ> = Ez Ezᵀ + M_inv
            EzzT[n] = np.outer(Ez_n,Ez_n) + M_n_inv
            
            Numerator += np.outer(P_nTcentered[n], Ez_n)
            
            
        Denominator = np.tensordot(P_nTP_n_ii, EzzT, axes=(0, 0))
        W_new = np.zeros((d, q))
        for i in range(d):
            if not np.any(Denominator[i]):  # degenerate case (feature i never observed)
                continue  # leave W_hat[i, :] = 0 
            W_new[i, :] = np.linalg.solve(Denominator[i], Numerator[i, :])
              
                   
        if (np.linalg.norm(W_new - W) / np.linalg.norm(W)) < tol:
            print(f'Converged. Iterations = {it+1}.')
            return W_new
        
        W = W_new
        
    print(f'Failed to converge. Iterations = {it+1}.')
    return W

## Synthetic Data

In [6]:
def generate_data(N, q, d, d_n, sparsity, projection_type, sigma2=(0.01, 0.05), max_iter=int(1e6)):
    """
    Generate synthetic centered data with general diagonal Σ_n
    
        N: number of samples
        q: latent dimension
        d: ambient dimension
        d_n: number of observed dimensions per sample, or inclusive range to draw from
        sparsity: fraction of zero entries in W; real in [0, 1) 
        projection_type: 'identity', 'selector', or 'scaled_selector'
        sigma2: fixed value or range for diagonal noise variances σ² 
        max_iter: max iterations to ensure W has full rank
    """
    if np.isscalar(d_n):
        d_n = np.full(N, d_n)
    else:
        d_n = np.random.randint(d_n[0], d_n[1]+1, size=N) 

    # full-rank = q matrix with sparsity_level % of entries 0
    for it in range(max_iter):
        W = np.random.randn(d, q)
        mask_entries = np.random.rand(d, q) < sparsity
        W[mask_entries] = 0.0
        if np.linalg.matrix_rank(W) == q:
            break
    else:
        raise ValueError(f"Failed to find full-rank W after {max_iter} iterations. Reduce sparsity level. ")

    Z = np.random.randn(N, q)
    mu = np.zeros(d)  # centered

    if projection_type == 'identity':
        P = [np.eye(d)[:d_n[n]] for n in range(N)]
    elif projection_type == 'selector':
        P = [np.eye(d)[np.random.choice(d, d_n[n], replace=False)] for n in range(N)]  # missing data
    elif projection_type == 'scaled_selector':
        P = [np.diag(np.random.randn(d))[np.random.choice(d, d_n[n], replace=False)] for n in range(N)]
    else:
        raise ValueError("Choose valid projection_type.")

    if np.isscalar(sigma2): 
        sigma2 = (sigma2, sigma2)
    Sigma_diag = [np.random.uniform(*sigma2, size=d_n[n]) for n in range(N)]
    eps = [np.random.randn(d_n[n]) * np.sqrt(Sigma_diag[n]) for n in range(N)]
    y = [P[n] @ (W @ Z[n]) + eps[n] for n in range(N)]

    dims = dict(N=N, q=q, d=d, d_n=d_n)
    
    return P, y, mu, Sigma_diag, Z, W, dims

## Evaluation

In [7]:
def subspace_err(W1, W2):
    """Compute Frobenius norm between projection matrices of two subspaces."""
    
    P1 = W1 @ np.linalg.pinv(W1.T @ W1) @ W1.T
    P2 = W2 @ np.linalg.pinv(W2.T @ W2) @ W2.T
    
    return np.linalg.norm(P1 - P2)

## Example Run

In [8]:
np.random.seed(3)

In [9]:
P, y, mu, Sigma_diag, Z, W, dims = generate_data(
    N=2000, 
    q=3, 
    d=5, 
    d_n=(5, 5), 
    sparsity = 0.7, 
    projection_type='selector', 
    sigma2 = 0.2
)

W_init = np.random.randn(*W.shape) # initialize W 

In [10]:
lam = 100  # adjust depending on data
W_em, *_ = fit_lin_sparse_ppca(P, y, mu, Sigma_diag, W_init, lam)

Converged. Iterations = 24.


In [11]:
W_base = fit_lin_ppca(P, y, mu, Sigma_diag, W_init)

Converged. Iterations = 23.


### Results

In [12]:
print("Model subspace error:", subspace_err(W_em, W))
print("Baseline subspace error:", subspace_err(W_base, W))

Model subspace error: 0.008233546748152342
Baseline subspace error: 0.04411720101191956
